In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Comparative Pipeline Benchmark: WITH Layer 1 vs WITHOUT Layer 1 (`models/combined_hierarchical_triage_pipeline.ipynb`)

This notebook evaluates and compares full **5-Class Probabilistic Triage Pipeline** performance on the **Holdout Test Set** (ratios parsed strictly from `config/triage_conf.json`), benchmarked on **Recall**, **Specificity**, **Balanced Accuracy**, and **ROC-AUC**:

### Layer Feature Definitions
- **Layer 1 (XGBoost ESI 1 Detector)**: 45 Features (19 raw inputs + 10 binary vital flags + 16 continuous deltas & ranges).
- **Layer 2 (Random Forest 3-Class Grouped Model)**: 29 Features (19 raw inputs + 10 binary vital anomaly flags; non-binary deltas/ranges removed).
- **Layer 3 (Dual Specialist LightGBM 2/3 + XGBoost 4/5)**: 29 Features (19 raw inputs + 10 binary vital anomaly flags; non-binary deltas/ranges removed).

### Architectural Configurations Evaluated
1. **MODE 1: WITH Layer 1 (Dedicated Binary XGBoost ESI 1 Detector)**:
   - Uses $P_1(\text{ESI 1})$ for ESI 1 and scales downstream non-ESI 1 branches by $(1 - P_1(\text{ESI 1}))$.
2. **MODE 2: WITHOUT Layer 1 (Layer 2 Predicts ESI 1 Directly)**:
   - Uses Layer 2's `"other"` class probability ($P_2(\text{other})$) directly for $P(\text{ESI 1})$.
   - Calculates $P(\text{ESI 2..5})$ directly from unscaled joint probability products.

### Target Benchmark Metrics Suite
Evaluates ONLY **Recall (Sensitivity)**, **Specificity**, **Balanced Accuracy**, and **ROC-AUC**.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(xgboost)
  library(ranger)
})
has_lgb <- requireNamespace("lightgbm", quietly = TRUE)
if (has_lgb) library(lightgbm)
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Master Pipeline Benchmark Initialized ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Full Dataset & Construct Features
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
# Construct Full Data Frame
df_full <- data.frame(
  # 19 Raw Inputs
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_last              = p_last,
  resp_last               = r_last,
  spo2_last               = o2_last,
  sbp_last                = s_last,
  pulse_min               = p_min,
  resp_min                = r_min,
  spo2_min                = o2_min,
  sbp_min                 = s_min,
  pulse_max               = p_max,
  resp_max                = r_max,
  spo2_max                = o2_max,
  sbp_max                 = s_max,
  
  # 10 Binary Vital Anomaly Flags
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0),
  
  # 16 Continuous Vital Delta & Range Features
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows (Remaining: %d)\n", initial_rows - nrow(df_full), nrow(df_full)))
# Stratified Test Partitioning (Parsed from config/triage_conf.json)
test_size <- config$training$test_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
test_df      <- df_full[-in_train_val, ]
cat(sprintf("Holdout Test Set Ready (from config test_size=%.4f): %d rows x %d cols\n", test_size, nrow(test_df), ncol(test_df)))
cat("Natural 5-Class Target Distribution (ESI 1 to 5):\n")
print(table(test_df$target_col))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Load Saved Model Artifacts from deploy/
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
path_esi1  <- file.path(deploy_dir, "xgboost_raw_esi1_extreme_model.rds")
path_rf    <- file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds")
path_ds    <- file.path(deploy_dir, "xgboost_esi23_esi45_extreme_model.rds")
path_lgb23 <- file.path(deploy_dir, "lightgbm_esi23_model.rds")
path_xgb45 <- file.path(deploy_dir, "xgboost_esi45_model.rds")
cat("Loading model artifacts...\n")
art_esi1  <- readRDS(path_esi1)
art_rf    <- readRDS(path_rf)
art_ds    <- readRDS(path_ds)
art_lgb23 <- if (file.exists(path_lgb23)) readRDS(path_lgb23) else NULL
art_xgb45 <- if (file.exists(path_xgb45)) readRDS(path_xgb45) else NULL
cat("  - Layer 1 Model (XGBoost ESI 1 Detector) Loaded.\n")
cat("  - Layer 2 Model (Random Forest 3-Class Grouped Model) Loaded.\n")
cat("  - Layer 3A Model (LightGBM ESI 2 vs 3 Specialist) Loaded.\n")
cat("  - Layer 3B Model (XGBoost ESI 4 vs 5 Specialist) Loaded.\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Run Inference WITH Layer 1 vs WITHOUT Layer 1
# ---------------------------------------------------------
# 1. Layer 1: XGBoost ESI 1 Detector Prediction (45 Features)
test_esi1_scaled <- predict(art_esi1$preproc, test_df)
feat_esi1        <- setdiff(names(test_esi1_scaled), "target_col")
dtest_esi1       <- xgb.DMatrix(data = as.matrix(test_esi1_scaled[, feat_esi1]))
p_esi1_L1        <- predict(art_esi1$model, dtest_esi1)
# 2. Layer 2: Random Forest 3-Class Grouped Model ('2_3', '4_5', 'other' [ESI 1]) (29 Features)
rf_feats         <- names(art_rf$preproc$mean)
test_rf_scaled   <- predict(art_rf$preproc, test_df[, c(rf_feats, intersect(names(test_df), c("gender", "cc_breathingdifficulty", "is_dyspnea_total", "is_dyspnea_moderate", "is_bradypnea", "is_tachypnea", "is_hypotension", "is_hypertension", "is_bradycardia_total", "is_bradycardia_moderate", "is_tachycardia_total", "is_tachycardia_moderate")))])
rf_raw_probs     <- predict(art_rf$model, data = test_rf_scaled)$predictions
p_rf_23    <- rf_raw_probs[, "2_3"]
p_rf_45    <- rf_raw_probs[, "4_5"]
p_rf_other <- rf_raw_probs[, "other"] # DIRECT LAYER 2 PREDICTION FOR ESI 1
# 3. Layer 3: Specialist Conditional Probabilities (29 Features)
binary_cols <- c("gender", "cc_breathingdifficulty",
                 "is_dyspnea_total", "is_dyspnea_moderate", "is_bradypnea", "is_tachypnea",
                 "is_hypotension", "is_hypertension", "is_bradycardia_total", "is_bradycardia_moderate",
                 "is_tachycardia_total", "is_tachycardia_moderate")
cont_cols_ds   <- names(art_ds$preproc$mean)
all_feats_ds   <- c(binary_cols, cont_cols_ds)
test_ds_scaled <- predict(art_ds$preproc, test_df[, c(cont_cols_ds, binary_cols)])
test_ds_x      <- as.matrix(test_ds_scaled[, all_feats_ds])
if (!is.null(art_lgb23) && art_lgb23$has_lgb) {
  p_esi2_given_23 <- predict(art_lgb23$model, test_ds_x)
} else if (art_ds$has_lgb) {
  p_esi2_given_23 <- predict(art_ds$model_lgb, test_ds_x)
} else {
  p_esi2_given_23 <- predict(art_ds$model_lgb, xgb.DMatrix(data = test_ds_x))
}
p_esi3_given_23 <- 1 - p_esi2_given_23
if (!is.null(art_xgb45)) {
  p_esi4_given_45 <- predict(art_xgb45$model, xgb.DMatrix(data = test_ds_x))
} else {
  p_esi4_given_45 <- predict(art_ds$model_xgb, xgb.DMatrix(data = test_ds_x))
}
p_esi5_given_45 <- 1 - p_esi4_given_45
# ---------------------------------------------------------
# MODE 1: WITH LAYER 1 (Overriding ESI 1 with Dedicated XGBoost Specialist)
# ---------------------------------------------------------
denom_rf <- p_rf_23 + p_rf_45
denom_rf[denom_rf == 0] <- 1
p_withL1_1 <- p_esi1_L1
p_withL1_2 <- (1 - p_esi1_L1) * (p_rf_23 / denom_rf) * p_esi2_given_23
p_withL1_3 <- (1 - p_esi1_L1) * (p_rf_23 / denom_rf) * p_esi3_given_23
p_withL1_4 <- (1 - p_esi1_L1) * (p_rf_45 / denom_rf) * p_esi4_given_45
p_withL1_5 <- (1 - p_esi1_L1) * (p_rf_45 / denom_rf) * p_esi5_given_45
probs_withL1 <- cbind(p_withL1_1, p_withL1_2, p_withL1_3, p_withL1_4, p_withL1_5)
colnames(probs_withL1) <- c("1", "2", "3", "4", "5")
# ---------------------------------------------------------
# MODE 2: WITHOUT LAYER 1 (ESI 1 Directly Calculated by Layer 2 'Other' Class)
# ---------------------------------------------------------
p_noL1_1 <- p_rf_other  # DIRECT LAYER 2 PREDICTION FOR ESI 1
p_noL1_2 <- p_rf_23 * p_esi2_given_23
p_noL1_3 <- p_rf_23 * p_esi3_given_23
p_noL1_4 <- p_rf_45 * p_esi4_given_45
p_noL1_5 <- p_rf_45 * p_esi5_given_45
probs_noL1 <- cbind(p_noL1_1, p_noL1_2, p_noL1_3, p_noL1_4, p_noL1_5)
colnames(probs_noL1) <- c("1", "2", "3", "4", "5")
cat(sprintf("Probability Sum Check (WITH Layer 1):    Mean = %.6f\n", mean(rowSums(probs_withL1))))
cat(sprintf("Probability Sum Check (WITHOUT Layer 1): Mean = %.6f\n", mean(rowSums(probs_noL1))))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Benchmark and Compare WITH Layer 1 vs WITHOUT Layer 1 (Recall, Specificity, BalAcc, ROC-AUC ONLY)
# ---------------------------------------------------------
act_fac <- factor(test_df$target_col, levels = c("1", "2", "3", "4", "5"))
eval_mode <- function(probs_mat, name_str) {
  pred_idx <- apply(probs_mat, 1, which.max)
  pred_fac <- factor(colnames(probs_mat)[pred_idx], levels = c("1", "2", "3", "4", "5"))
  
  cm  <- confusionMatrix(pred_fac, act_fac)
  
  rec_by_cls     <- as.numeric(cm$byClass[, "Sensitivity"])
  spec_by_cls    <- as.numeric(cm$byClass[, "Specificity"])
  bal_acc_by_cls <- as.numeric(cm$byClass[, "Balanced Accuracy"])
  
  rec_by_cls[is.na(rec_by_cls)]         <- 0
  spec_by_cls[is.na(spec_by_cls)]       <- 0
  bal_acc_by_cls[is.na(bal_acc_by_cls)] <- 0
  
  roc_auc_by_cls <- sapply(1:5, function(i) {
    cls_name <- levels(act_fac)[i]
    act_bin  <- ifelse(act_fac == cls_name, 1, 0)
    r_obj    <- tryCatch(pROC::roc(act_bin, probs_mat[, i]), error = function(e) NULL)
    if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
  })
  macro_rec     <- mean(rec_by_cls)
  macro_spec    <- mean(spec_by_cls)
  macro_bal_acc <- mean(bal_acc_by_cls)
  macro_auc     <- mean(roc_auc_by_cls, na.rm = TRUE)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   5-CLASS BENCHMARK: %s\n", toupper(name_str)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Macro Recall (Sens)     : %.4f\n", macro_rec))
  cat(sprintf("  Macro Specificity       : %.4f\n", macro_spec))
  cat(sprintf("  Macro Balanced Accuracy : %.4f\n", macro_bal_acc))
  cat(sprintf("  Macro ROC-AUC           : %.4f\n", macro_auc))
  cat(sprintf("============================================================\n\n"))
  print(cm$table)
  cat("\n\n")
  
  return(list(
    rec = macro_rec, spec = macro_spec, bal_acc = macro_bal_acc, auc = macro_auc
  ))
}
res_withL1 <- eval_mode(probs_withL1, "With Layer 1 (Dedicated XGBoost ESI 1 Detector)")
res_noL1   <- eval_mode(probs_noL1,   "Without Layer 1 (ESI 1 Predicted Directly by Layer 2 'Other' Class)")
# Write CSV Summary Report
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
comp_report_df <- data.frame(
  Configuration           = c("With_Layer1", "Without_Layer1_Layer2Direct"),
  Macro_Recall            = round(c(res_withL1$rec,     res_noL1$rec), 4),
  Macro_Specificity       = round(c(res_withL1$spec,    res_noL1$spec), 4),
  Macro_Balanced_Accuracy = round(c(res_withL1$bal_acc, res_noL1$bal_acc), 4),
  Macro_ROC_AUC           = round(c(res_withL1$auc,     res_noL1$auc), 4)
)
write.csv(comp_report_df, file = file.path(reports_dir, "combined_pipeline_test_report.csv"), row.names = FALSE)
cat("Layer 1 Comparison Test Report written to: reports/combined_pipeline_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Diagnostic Plots (Metrics Bar Chart for Recall, Specificity, BalAcc, ROC-AUC ONLY)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_comp <- data.frame(
  Metric = rep(c("Recall", "Specificity", "Balanced_Acc", "ROC_AUC"), 2),
  Configuration = c(rep("With Layer 1", 4), rep("Without Layer 1 (Layer 2 Direct)", 4)),
  Score = c(
    res_withL1$rec, res_withL1$spec, res_withL1$bal_acc, res_withL1$auc,
    res_noL1$rec,   res_noL1$spec,   res_noL1$bal_acc,   res_noL1$auc
  )
)
p_bar <- ggplot(metrics_comp, aes(x = Metric, y = Score, fill = Configuration)) +
  geom_bar(stat = "identity", position = "dodge", width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.6), vjust = -0.3, size = 3.2, fontface = "bold") +
  theme_minimal() +
  scale_fill_manual(values = c("With Layer 1" = "#2b5c8f", "Without Layer 1 (Layer 2 Direct)" = "#e07a5f")) +
  labs(title = "Full Triage Pipeline Benchmark: WITH Layer 1 vs. WITHOUT Layer 1",
       subtitle = "Comparing Primary Performance Metrics on Holdout Test Set",
       y = "Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13, hjust = 0.5),
        plot.subtitle = element_text(size = 9.5, hjust = 0.5),
        axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "top")
ggsave(file.path(plots_dir, "combined_pipeline_metrics_barchart.png"), plot = p_bar, width = 9.5, height = 5.0, dpi = 300)
cat("Layer 1 Comparison Bar Chart saved to: plots/combined_pipeline_metrics_barchart.png\n")
p_bar